# Comparaison des solveurs pour le Meeting Scheduling Problem

Ce notebook analyse les performances de trois solveurs (ACE, Choco, OR-Tools) avec et sans optimisation (minimisation du makespan).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results_meeting")

FILES = {
    "ACE": "results_ace.json",
    "ACE_OPT": "results_ace_opt.json",
    "CHOCO": "results_choco.json",
    "CHOCO_OPT": "results_choco_opt.json",
    "OR_TOOLS": "results_ortools.json",
    "OR_TOOLS_OPT": "results_ortools_opt.json",
}

In [ ]:
def load_results(file_path):
    """Charge un fichier JSON et retourne (instances, solver, total_time)."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
        if "instances" in data:
            instances = data["instances"]
            solver = data.get("solver", "unknown")
            total_time = data.get("total_time_seconds", 0.0)
        else:
            instances = data
            solver = "unknown"
            total_time = sum(inst.get("time_seconds", 0) for inst in instances)
        return instances, solver, total_time
    except FileNotFoundError:
        print(f"Fichier non trouvé : {file_path}")
        return [], "unknown", 0.0
    except json.JSONDecodeError:
        print(f"Erreur de décodage JSON : {file_path}")
        return [], "unknown", 0.0

def analyze_instances(instances):
    """Analyse une liste d'instances et retourne les métriques."""
    solved = 0
    times = []
    for inst in instances:
        if inst.get("status") == "solved":
            solved += 1
        t = inst.get("time_seconds", inst.get("time", 0.0))
        times.append(t)
    avg_time = sum(times) / len(times) if times else 0.0
    return {
        "solved": solved,
        "unsolved": len(instances) - solved,
        "avg_time": avg_time,
        "times": times
    }

In [ ]:
results = {}
for name, filename in FILES.items():
    filepath = RESULTS_DIR / filename
    instances, solver, total_time = load_results(filepath)
    analysis = analyze_instances(instances)
    results[name] = {
        "solver": solver,
        "total_time": total_time,
        "solved": analysis["solved"],
        "unsolved": analysis["unsolved"],
        "avg_time": analysis["avg_time"],
    }
    print(f"Chargé {name}: {analysis['solved']} solved / {analysis['unsolved']} unsolved")

## Tableau récapitulatif

In [ ]:
df = pd.DataFrame([
    {
        "Solver": name,
        "Solved": r["solved"],
        "Unsolved": r["unsolved"],
        "Avg. Time (s)": round(r["avg_time"], 3),
        "Total Time (s)": round(r["total_time"], 3)
    }
    for name, r in results.items()
])
print("\n===== COMPARAISON GLOBALE =====\n")
print(df.to_string(index=False))

## Graphiques comparatifs

In [ ]:
colors = {"ACE": "#1f77b4", "ACE_OPT": "#1f77b4",
          "CHOCO": "#ff7f0e", "CHOCO_OPT": "#ff7f0e",
          "OR_TOOLS": "#2ca02c", "OR_TOOLS_OPT": "#2ca02c"}

solvers = ["ACE", "ACE_OPT", "CHOCO", "CHOCO_OPT", "OR_TOOLS", "OR_TOOLS_OPT"]

# 1. Nombre d'instances résolues
plt.figure(figsize=(10, 5))
solved_vals = [results[s]["solved"] for s in solvers]
bars = plt.bar(solvers, solved_vals, color=[colors[s] for s in solvers])
plt.title("Number of Solved Instances (out of 27)", fontsize=14)
plt.ylabel("Solved instances")
plt.ylim(0, 28)
for bar, val in zip(bars, solved_vals):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val),
             ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Temps moyen par solveur
plt.figure(figsize=(10, 5))
avg_times = [results[s]["avg_time"] for s in solvers]
bars = plt.bar(solvers, avg_times, color=[colors[s] for s in solvers])
plt.title("Average Solving Time per Instance (solved instances)", fontsize=14)
plt.ylabel("Average time (seconds)")
for bar, val in zip(bars, avg_times):
    if val > 0:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{val:.2f}", ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Temps total d'exécution
plt.figure(figsize=(10, 5))
total_times = [results[s]["total_time"] for s in solvers]
bars = plt.bar(solvers, total_times, color=[colors[s] for s in solvers])
plt.title("Total Execution Time (all instances)", fontsize=14)
plt.ylabel("Total time (seconds)")
for bar, val in zip(bars, total_times):
    if val > 0:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}", ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Impact de l'optimisation (sans vs avec minimisation)

In [ ]:
base_solvers = ["ACE", "CHOCO", "OR_TOOLS"]
opt_labels = ["Without optimization", "With optimization"]

for base in base_solvers:
    without = results[base]
    with_opt = results[f"{base}_OPT"]
    plt.figure(figsize=(6, 4))
    x = np.arange(2)
    plt.bar(x, [without["solved"], with_opt["solved"]], width=0.6,
            color=["#1f77b4", "#ff7f0e"], alpha=0.8)
    plt.xticks(x, opt_labels)
    plt.title(f"{base} : Solved instances (with/without optimization)")
    plt.ylabel("Number of solved instances")
    for i, val in enumerate([without["solved"], with_opt["solved"]]):
        plt.text(i, val + 0.5, str(val), ha='center', va='bottom')
    plt.tight_layout()
    plt.show()

## Détail par instance (carte de chaleur des temps)

In [ ]:
def get_times_by_instance(filepath):
    instances, _, _ = load_results(filepath)
    return {inst["instance_id"]: inst.get("time_seconds", 0.0) for inst in instances}

times_data = {}
for name, filename in FILES.items():
    filepath = RESULTS_DIR / filename
    times_data[name] = get_times_by_instance(filepath)

all_ids = list(range(1, 28))
df_times = pd.DataFrame(index=all_ids)

for name in solvers:
    df_times[name] = [times_data[name].get(i, None) for i in all_ids]

df_times_display = df_times.fillna(10.0)

plt.figure(figsize=(14, 8))
im = plt.imshow(df_times_display.values.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=10)
plt.colorbar(im, label="Time (seconds)")
plt.yticks(range(len(solvers)), solvers)
plt.xticks(range(len(all_ids)), all_ids, rotation=90)
plt.title("Solving time per instance (seconds) – Red = >10s or unsolved")
plt.xlabel("Instance ID")
plt.ylabel("Solver")
plt.tight_layout()
plt.show()

## Conclusion
- Les tableaux montrent le nombre d'instances résolues et les temps moyens.
- Les graphiques comparent l'impact de l'optimisation (minimisation du makespan) pour chaque solveur.
- La carte de chaleur permet d'identifier les instances difficiles pour chaque solveur.